In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 데이터 로드
train = pd.read_csv('../data/raw/UNSW_NB15_training-set.csv')
test = pd.read_csv('../data/raw/UNSW_NB15_testing-set.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

/var/folders/s9/5g6t7vwn40ngmg0cqw8zz1nh0000gn/T/ipykernel_18794/3090506353.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Train shape: (82332, 45)
Test shape: (175341, 45)


In [2]:
# 불필요한 컬럼 제거
# id는 식별자라 모델 학습에 불필요
# attack_cat은 label로 대체 가능
drop_cols = ['id', 'attack_cat']

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

print("Train shape after drop:", train.shape)
print("Test shape after drop:", test.shape)

Train shape after drop: (82332, 43)
Test shape after drop: (175341, 43)


In [ ]:
# 범주형 피처 인코딩
# proto, service, state 이 3개가 문자열이라 숫자로 바꿔줘야 모델이 읽을 수 있기 때문
cat_cols = ['proto', 'service', 'state']

le = LabelEncoder()

for col in cat_cols:
    # train이랑 test 합쳐서 인코딩해야 일관성 유지가능
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

print("인코딩 완료!")
print(train[cat_cols].head())

인코딩 완료!
   proto  service  state
0    119        0      5
1    119        0      5
2    119        0      5
3    119        0      5
4    119        0      5


In [4]:
# 피처와 타겟 분리
X_train = train.drop(columns=['label'])
y_train = train['label']

X_test = test.drop(columns=['label'])
y_test = test['label']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train 분포:\n", y_train.value_counts())

X_train shape: (82332, 42)
X_test shape: (175341, 42)
y_train 분포:
 label
1    45332
0    37000
Name: count, dtype: int64


In [ ]:
# 스케일링
# 피처마다 범위가 달라서 통일시켜줘야 하기 때문
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("스케일링 완료!")
print("X_train 평균 (첫 5개 피처):", X_train_scaled[:, :5].mean(axis=0).round(3))

스케일링 완료!
X_train 평균 (첫 5개 피처): [-0.  0.  0.  0. -0.]


In [6]:
import os

# 전처리 데이터 저장
os.makedirs('../data/processed', exist_ok=True)

# numpy 배열로 저장
np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy', y_test.values)

# 스케일러랑 인코더는 나중에 서빙할 때 필요해서 저장
import joblib
joblib.dump(scaler, '../data/processed/scaler.joblib')

print("전처리 데이터 저장 완료!")

전처리 데이터 저장 완료!
